# 02 · Preprocessing + RUL
**Owner:** Wijayagurusinghe · **Branch:** `data/02-preprocessing-rul`

Keep only cells that actually reach end-of-life, add the RUL label (`RUL = cycle_life − cycle`), do safe cleaning, and save `preprocessed.parquet`.


In [ ]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

# Try Google Drive (data persists across sessions). If auth fails, fall back
# to local Colab storage so you are never blocked.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


In [ ]:
import pickle, pandas as pd
from src.preprocessing import (build_summary_table, add_rul, reached_eol,
    basic_clean, clean_anomalies, quality_report, print_report, EOL_AH)
bat_dict = pickle.load(open(os.path.join(DATA_DIR,'bat_dict.pkl'),'rb'))


### Drop cells that never reach EOL (≤ 0.88 Ah)


In [ ]:
eol = reached_eol(bat_dict)
bad = [c for c,ok in eol.items() if not ok]
print('cells not reaching EOL:', bad or 'none')
bat_dict = {c:d for c,d in bat_dict.items() if eol[c]}
print('cells kept:', len(bat_dict))


### Build table + RUL label + clean
Safe cleaning (drop duplicates + formation cycle 1), then repair measurement anomalies: drop physically-impossible capacity rows (QD/QC > 1.2 Ah) and fix IR<=0 sensor dropouts (forward/back-fill within cell).


In [ ]:
df = add_rul(build_summary_table(bat_dict))
clean = basic_clean(df, drop_first_cycle=True)
clean, anom = clean_anomalies(clean)
print('anomaly report:', anom)
print('rows:', len(df), '->', len(clean), '| cells:', clean['cell'].nunique())
clean.to_parquet(os.path.join(DATA_DIR,'preprocessed.parquet'), index=False)
print_report(quality_report(clean))


### Distribution notes (document for the report)
Batch lifetimes differ (batch 2 is short-lived) and RUL is right-skewed — noted as limitations; the split (nb 04) is batch-stratified to compensate.


In [ ]:
print('data points:', len(clean), '| cells:', clean['cell'].nunique())
print('batch median cycle_life:', clean.groupby('batch')['cycle_life'].median().round(0).to_dict())
print('RUL quantiles:', clean['RUL'].quantile([.1,.25,.5,.75,.9]).round(0).to_dict())


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,4))
ax[0].hist(clean['RUL'],bins=50); ax[0].set_title('RUL'); ax[0].set_xlabel('cycles')
for c in list(clean['cell'].unique())[:15]:
    g=clean[clean['cell']==c]; ax[1].plot(g['cycle'],g['QD'],lw=.6)
ax[1].set_title('capacity fade'); ax[1].set_xlabel('cycle'); ax[1].set_ylabel('QD (Ah)')
plt.tight_layout(); plt.show()
